# Laboratorio 19. Proyecto integrador de Deep Learning

**Pregunta central:** ¿Una CNN aprovecha mejor la estructura espacial que un MLP cuando ambos reciben los mismos ejemplos, particiones y presupuesto de entrenamiento?

## Objetivos

- Construir un problema reproducible de clasificación de imágenes pequeñas sin descargas.
- Separar train, validación y test antes de ajustar; normalizar con estadísticas de train.
- Comparar un baseline, un MLP y una CNN con arquitectura compacta, curvas de aprendizaje y análisis de errores.
- Elegir con validación y reservar test para la comparación final.

## Prerrequisitos y conexión

Integra tensores y autograd, entrenamiento, generalización, convoluciones y lectura de errores (laboratorios 01–14). Cada paso fundamental queda visible; no se requiere un dataset externo.

## Teoría breve

Un clasificador parametrizado produce logits $z=f_\theta(x)$ y minimiza entropía cruzada. La CNN comparte filtros en el espacio; el MLP aplana la imagen. Una comparación es informativa solo si se mantienen datos y protocolo, aunque no iguala exactamente el número de parámetros.

## Problema y datos

Clasificamos imágenes sintéticas de 16×16 con una cruz o un anillo, perturbadas por traslación y ruido. La etiqueta se genera al crear cada patrón. La tarea prueba el flujo de trabajo, no el rendimiento en imágenes naturales.

## Plan de evaluación

Usaremos divisiones estratificadas 70/15/15, exactitud y matriz de confusión. El baseline mayoritario y los modelos usarán el mismo split. Se escogerá la época con mejor exactitud de validación; test se abrirá al final.

### Preparación del entorno
Importamos NumPy y PyTorch, fijamos semillas y elegimos CPU o GPU de forma opcional.


In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch import nn
import torch.nn.functional as F

SEED = 2026
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
plt.rcParams['figure.figsize'] = (7, 4)
print('PyTorch:', torch.__version__, '| dispositivo:', device)


**Resultado e interpretación.** La celda confirma el entorno utilizado. El resto del laboratorio está diseñado para CPU; una GPU disponible se utiliza sin ser requisito.


### Crear secuencias tokenizadas
Separamos ejemplos antes de entrenar y reservamos validación para decisiones.


In [ ]:
rng = np.random.default_rng(SEED)
images, labels = [], []
image_size = 16
for label in range(2):
    for _ in range(400):
        image = np.zeros((image_size, image_size), dtype=np.float32)
        if label == 0:
            image[7:9, 2:14] = 1.0
            image[2:14, 7:9] = 1.0
        else:
            rows, columns = np.ogrid[:image_size, :image_size]
            radius = np.sqrt((rows - 7.5) ** 2 + (columns - 7.5) ** 2)
            image[(radius >= 4) & (radius <= 6)] = 1.0
        image = np.roll(image, shift=(rng.integers(-1, 2), rng.integers(-1, 2)), axis=(0, 1))
        image += rng.normal(0, 0.18, image.shape).astype(np.float32)
        images.append(np.clip(image, 0, 1))
        labels.append(label)
images = np.asarray(images, dtype=np.float32)
labels = np.asarray(labels, dtype=np.int64)
train_indices, validation_indices, test_indices = [], [], []
for label in range(2):
    class_indices = np.flatnonzero(labels == label)
    rng.shuffle(class_indices)
    train_indices.extend(class_indices[:280])
    validation_indices.extend(class_indices[280:340])
    test_indices.extend(class_indices[340:])
rng.shuffle(train_indices); rng.shuffle(validation_indices); rng.shuffle(test_indices)
X = torch.from_numpy(images[:, None]).to(device)
y = torch.from_numpy(labels).to(device)
train_idx = torch.tensor(train_indices, dtype=torch.long, device=device)
val_idx = torch.tensor(validation_indices, dtype=torch.long, device=device)
test_idx = torch.tensor(test_indices, dtype=torch.long, device=device)
train_mean = X[train_idx].mean()
train_std = X[train_idx].std().clamp_min(1e-6)
X = (X - train_mean) / train_std
baseline_class = torch.bincount(y[train_idx]).argmax()
baseline_validation = (y[val_idx] == baseline_class).float().mean().item()
baseline_test = (y[test_idx] == baseline_class).float().mean().item()
print('tamaños train/validación/test:', len(train_idx), len(val_idx), len(test_idx))
print('media/desviación train:', round(train_mean.item(), 4), round(train_std.item(), 4))
print('baseline exactitud validación/test:', round(baseline_validation, 3), round(baseline_test, 3))
fig, axes = plt.subplots(1, 6, figsize=(9, 2))
for ax, index in zip(axes, train_idx[:6].tolist()):
    ax.imshow(X[index, 0].detach().cpu(), cmap='gray')
    ax.set_title(f'clase {y[index].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

**Resultado e interpretación.** La etiqueta se deriva de la secuencia y el vocabulario tiene tamaño pequeño. El baseline mayoritario contextualiza la exactitud.


### Entrenar modelos con el mismo protocolo

El MLP aplana la imagen y la CNN conserva vecindades mediante convoluciones. Usamos igual partición, pérdida, tamaño de lote y máximo de épocas. Guardamos el estado con mejor exactitud de validación y mostramos train/validación; las métricas de test se calculan en la celda siguiente.

In [ ]:
def make_mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(image_size * image_size, 48), nn.ReLU(), nn.Linear(48, 2))


def make_cnn():
    return nn.Sequential(
        nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(16 * 4 * 4, 2),
    )

model_factories = {'MLP': make_mlp, 'CNN': make_cnn}
trained_models, learning_curves, validation_results = {}, {}, {}
for model_number, (model_name, make_model) in enumerate(model_factories.items()):
    torch.manual_seed(SEED + model_number)
    model = make_model().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.004)
    history = {'train_loss': [], 'validation_loss': [], 'validation_accuracy': []}
    best_accuracy = -1.0
    best_state = None
    for epoch in range(14):
        model.train()
        shuffled_indices = train_idx[torch.randperm(len(train_idx), device=device)]
        total_loss = 0.0
        for batch_indices in shuffled_indices.split(64):
            logits = model(X[batch_indices])
            loss = F.cross_entropy(logits, y[batch_indices])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_indices)
        history['train_loss'].append(total_loss / len(train_idx))
        model.eval()
        with torch.no_grad():
            validation_logits = model(X[val_idx])
            validation_loss = F.cross_entropy(validation_logits, y[val_idx]).item()
            validation_accuracy = (validation_logits.argmax(dim=1) == y[val_idx]).float().mean().item()
        history['validation_loss'].append(validation_loss)
        history['validation_accuracy'].append(validation_accuracy)
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_state = {key: value.detach().clone() for key, value in model.state_dict().items()}
    model.load_state_dict(best_state)
    model.eval()
    trained_models[model_name] = model
    learning_curves[model_name] = history
    validation_results[model_name] = best_accuracy
    print(model_name, '| mejor exactitud validación:', round(best_accuracy, 3))

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for model_name, history in learning_curves.items():
    axes[0, 0].plot(history['train_loss'], label=model_name)
    axes[0, 1].plot(history['validation_loss'], label=model_name)
    axes[1, 0].plot(history['validation_accuracy'], label=model_name)
for ax, title, ylabel in zip(axes.flat[:3], ['Pérdida train', 'Pérdida validación', 'Exactitud validación'], ['Cross-entropy', 'Cross-entropy', 'Exactitud']):
    ax.set(xlabel='Época', ylabel=ylabel, title=title)
    ax.legend()
axes[1, 1].axis('off')
plt.tight_layout()
plt.show()

### Evaluación final y análisis de errores

Los checkpoints ya están elegidos con validación. Calculamos ahora exactitud y matriz de confusión en test para el baseline, el MLP y la CNN. Después inspeccionamos ejemplos mal clasificados por la CNN y sus primeros mapas de características; si no hay errores, mostramos casos de test correctamente clasificados.

In [ ]:
test_results = {'Baseline': {'accuracy': baseline_test}}
confusion_matrices = {}
for model_name, model in trained_models.items():
    with torch.no_grad():
        test_logits = model(X[test_idx])
        test_predictions = test_logits.argmax(dim=1)
    test_accuracy = (test_predictions == y[test_idx]).float().mean().item()
    confusion = torch.bincount(2 * y[test_idx] + test_predictions, minlength=4).reshape(2, 2)
    test_results[model_name] = {'accuracy': test_accuracy}
    confusion_matrices[model_name] = confusion.cpu().numpy()
    print(model_name, '| exactitud test:', round(test_accuracy, 3))
print('Baseline | exactitud test:', round(baseline_test, 3))

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, model_name in zip(axes, ['MLP', 'CNN']):
    ax.imshow(confusion_matrices[model_name], cmap='Blues')
    for row in range(2):
        for column in range(2):
            ax.text(column, row, int(confusion_matrices[model_name][row, column]), ha='center', va='center')
    ax.set(xticks=[0, 1], yticks=[0, 1], xlabel='Predicción', ylabel='Real', title=f'{model_name}: matriz de confusión')
plt.tight_layout()
plt.show()

cnn_model = trained_models['CNN']
with torch.no_grad():
    wrong_mask = cnn_model(X[test_idx]).argmax(dim=1) != y[test_idx]
wrong_indices = test_idx[wrong_mask]
example_indices = wrong_indices[:6] if len(wrong_indices) else test_idx[:6]
print('Errores CNN en test:', len(wrong_indices), 'de', len(test_idx))
fig, axes = plt.subplots(1, len(example_indices), figsize=(10, 2))
for ax, index in zip(np.atleast_1d(axes), example_indices):
    with torch.no_grad():
        predicted_label = cnn_model(X[index:index+1]).argmax(dim=1).item()
    ax.imshow(X[index, 0].detach().cpu(), cmap='gray')
    ax.set_title(f'real {y[index].item()} / pred. {predicted_label}')
    ax.axis('off')
plt.tight_layout()
plt.show()

with torch.no_grad():
    feature_maps = cnn_model[:2](X[test_idx[:1]]).squeeze(0).cpu().numpy()
fig, axes = plt.subplots(1, 4, figsize=(8, 2))
for ax, feature_map in zip(axes, feature_maps[:4]):
    ax.imshow(feature_map, cmap='magma')
    ax.axis('off')
axes[0].set_title('Mapas CNN')
plt.tight_layout()
plt.show()

## Interpretación, limitaciones y conclusiones

En la ejecución actual, consulta las exactitudes y matrices impresas para comparar los modelos. Si MLP y CNN obtienen valores similares, la tarea sintética puede ser demasiado sencilla para distinguir sus sesgos inductivos; no concluyas que ambos son equivalentes en imágenes naturales. Una matriz de confusión vacía en errores tampoco prueba robustez fuera de esta distribución.

La comparación controla datos, particiones, pérdida y épocas, pero no número de parámetros ni búsqueda exhaustiva de hiperparámetros. La generación sintética representa dos patrones simples, no variación semántica real. La validación seleccionó la época; el test solo mide la configuración escogida. Para una conclusión más fuerte se necesitarían varias semillas, datos más diversos y un protocolo acordado antes de experimentar.

## Ejercicios

1. **Guiado:** reduce el conjunto de entrenamiento a 20 imágenes por clase, conservando validación y test; compara curvas y errores.
2. **Independiente:** añade rotaciones pequeñas como aumento de datos y analiza qué patrones de error cambian.
3. **Desafío:** diseña una comparación de al menos cinco semillas y reporta media y dispersión de exactitud de validación; no uses test para escoger arquitectura.

## Extensiones

Sustituye los generadores sintéticos por un dataset pequeño con licencia documentada y conserva este flujo como alternativa sin red. Antes de concluir, examina ejemplos mal clasificados y contrasta mapas de características con las imágenes de entrada.